<a href="https://colab.research.google.com/github/aicha-bakayoko/DI-BOOTCAMP/blob/main/W4S5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Load the Dataset

First, we need to load the customer churn dataset into a pandas DataFrame. I'll assume the data is available in a CSV file. If your data is from a different source or has a different path, please update the code accordingly.

In [ ]:
import pandas as pd

# Adjust the path if your CSV file is located elsewhere
try:
    df = pd.read_csv('/tmp/churn_data.csv')
except FileNotFoundError:
    print("Error: 'churn_data.csv' not found. Please upload the file or provide the correct path.")
    print("For example, if the file is in your Google Drive, you might need to mount your Drive first:")
    print("from google.colab import drive")
    print("drive.mount('/content/drive')")
    print("df = pd.read_csv('/content/drive/MyDrive/path/to/churn_data.csv')")

### Display the First Few Rows

Let's inspect the first few rows of the DataFrame to understand its structure and content.

In [ ]:
if 'df' in locals(): # Check if df was successfully created
    display(df.head())

### Initial Data Exploration and Preprocessing

Let's get a general understanding of the dataset, including data types, non-null values, and memory usage. This helps in identifying potential issues like incorrect data types or missing values.

In [ ]:
if 'df' in locals():
    print("\n--- DataFrame Info ---")
    df.info()

Next, let's look at the descriptive statistics of the numerical columns. This will provide insights into the central tendency, dispersion, and shape of the distribution of our numerical features.

In [ ]:
if 'df' in locals():
    print("\n--- Descriptive Statistics ---")
    display(df.describe())

It's crucial to identify and handle missing values. Let's check the count of missing values for each column.

In [ ]:
if 'df' in locals():
    print("\n--- Missing Values Count ---")
    display(df.isnull().sum())

Let's also check the unique values and their counts for categorical columns, as this will be important for encoding them later. We'll identify columns with object dtype or a limited number of unique values as potential categorical features.

In [ ]:
if 'df' in locals():
    print("\n--- Unique Values in Categorical Columns ---")
    categorical_cols = df.select_dtypes(include='object').columns
    if categorical_cols.empty:
        print("No object-type columns found. Checking other potential categorical columns.")
        # Also check for numerical columns with few unique values
        for col in df.columns:
            if df[col].nunique() < 20 and df[col].dtype != 'float64': # Heuristic for potentially categorical numerical columns
                print(f"\nColumn '{col}':")
                display(df[col].value_counts())
    else:
        for col in categorical_cols:
            print(f"\nColumn '{col}':")
            display(df[col].value_counts())

Finally, based on common practices in churn prediction, columns like `CustomerId`, `Surname`, and `RowNumber` are typically identifiers and do not contribute to the predictive power of the model. Let's drop them.

In [ ]:
if 'df' in locals():
    columns_to_drop = ['RowNumber', 'CustomerId', 'Surname']
    existing_columns_to_drop = [col for col in columns_to_drop if col in df.columns]

    if existing_columns_to_drop:
        df = df.drop(columns=existing_columns_to_drop)
        print(f"Dropped columns: {existing_columns_to_drop}")
    else:
        print("No common identifier columns (RowNumber, CustomerId, Surname) found to drop.")

    print("\n--- DataFrame after dropping identifier columns ---")
    df.info()

### Feature Engineering and Data Transformation

To prepare our data for machine learning, we need to convert categorical features into a numerical format. One-hot encoding is a suitable method for nominal categorical variables like 'Geography' and 'Gender'.

First, let's identify the categorical features.

In [ ]:
if 'df' in locals():
    # Identify categorical columns for one-hot encoding
    # Assuming 'Geography' and 'Gender' are the main categorical features based on typical churn datasets
    categorical_features = df.select_dtypes(include='object').columns.tolist()
    print(f"Categorical features identified for encoding: {categorical_features}")

    if categorical_features:
        df_encoded = pd.get_dummies(df, columns=categorical_features, drop_first=True)
        print("\n--- DataFrame after One-Hot Encoding ---")
        display(df_encoded.head())
        df_encoded.info()
    else:
        df_encoded = df.copy()
        print("No object-type categorical features found for encoding.")
        print("Using original DataFrame as 'df_encoded'.")

### Define Features (X) and Target (y)

Our goal is to predict customer churn, which is represented by the 'Exited' column. This will be our target variable (`y`). All other relevant columns will be used as features (`X`).

In [ ]:
if 'df_encoded' in locals() and 'Exited' in df_encoded.columns:
    X = df_encoded.drop('Exited', axis=1)
    y = df_encoded['Exited']

    print("\n--- Features (X) ---")
    display(X.head())
    print("\n--- Target (y) ---")
    display(y.head())
else:
    print("Cannot define X and y. Ensure 'df_encoded' exists and contains an 'Exited' column.")

### Split Data into Training and Testing Sets

Before training any model, it's essential to split the dataset into training and testing sets. The training set will be used to train the model, and the testing set will be used to evaluate its performance on unseen data. A common split ratio is 80% for training and 20% for testing.

In [ ]:
from sklearn.model_selection import train_test_split

if 'X' in locals() and 'y' in locals():
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    print(f"X_train shape: {X_train.shape}")
    print(f"X_test shape: {X_test.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"y_test shape: {y_test.shape}")
else:
    print("X or y not defined. Cannot split data.")

### Feature Scaling

Many machine learning algorithms perform better when numerical input variables are scaled to a standard range. We'll use `StandardScaler` to standardize features by removing the mean and scaling to unit variance. It's crucial to fit the scaler only on the training data to prevent data leakage.

In [ ]:
from sklearn.preprocessing import StandardScaler

if 'X_train' in locals() and 'X_test' in locals():
    scaler = StandardScaler()

    # Identify numerical columns for scaling
    # All columns in X_train are numerical after one-hot encoding, except possibly if `drop_first=False` was used
    # and original numerical columns were preserved.
    # For robustness, we can specifically select numerical types.
    numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns

    if not numerical_cols.empty:
        X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
        X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

        print("\n--- X_train after scaling (first 5 rows) ---")
        display(X_train.head())
        print("\n--- X_test after scaling (first 5 rows) ---")
        display(X_test.head())
    else:
        print("No numerical columns found for scaling in X_train/X_test.")
else:
    print("X_train or X_test not defined. Cannot perform feature scaling.")